<a href="https://colab.research.google.com/github/DeepthiManthapuram/Deep_Learning/blob/main/Positional_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention

# Input sentence

In [ ]:
sentence = ["I love deep learning"]
print(sentence)

['I love deep learning']


# Tokenization

In [ ]:
vectorizer = TextVectorization(output_mode='int', output_sequence_length=4)
vectorizer.adapt(sentence)

tokens = vectorizer(sentence)

print("Vocabulary:")
print(vectorizer.get_vocabulary())

print("Tokens:")
print(tokens.numpy())

Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:
[[4 2 5 3]]


# Word Embeddings

In [ ]:
embedding_dim=8

embedding_layer=Embedding(input_dim=len(vectorizer.get_vocabulary()),output_dim=embedding_dim)
word_embeddings=embedding_layer(tokens)

print("Word Embeddings:")
print(word_embeddings.numpy())

Word Embeddings:
[[[ 0.01029658  0.01769951 -0.03379195 -0.03439786 -0.03747253
   -0.01098936  0.04731724  0.01667133]
  [ 0.04544257  0.02325678 -0.02398947 -0.01038095 -0.04031844
    0.02530862  0.03206922  0.03444557]
  [-0.01063495 -0.01728457 -0.03438312 -0.04323583  0.03814309
   -0.01089318 -0.0103546   0.02940065]
  [-0.00438626 -0.04382576  0.02257687 -0.01134586  0.04855028
   -0.00241845  0.01194339  0.02575121]]]


# Positional Encoding Function

In [ ]:
def positional_encoding(max_position,d_model):
  positions=np.arange(max_position)[:,np.newaxis ]
  dimensions=np.arange(d_model)[np.newaxis,:]

  angle_rates=1/np.power(10000,(2*(dimensions//2))/tf.cast(d_model, dtype=tf.float32))

  angle_rads=positions * angle_rates

  PE=np.zeros((max_position,d_model))

  PE[:,0 :: 2]=np.sin(angle_rads[:,0 :: 2])
  PE[:,1 :: 2]=np.cos(angle_rads[:,1 :: 2])

  return tf.cast(PE,dtype=tf.float32)

# Generate Positional Encoding

In [ ]:
input_sequence = word_embeddings + PE

print("Input Sequence (Word Embeddings + Positional Encoding):")
print(input_sequence.numpy())

Input Sequence (Word Embeddings + Positional Encoding):
[[[ 0.01029658  1.0176995  -0.03379195  0.96560216 -0.03747253
    0.98901063  0.04731724  1.0166713 ]
  [ 0.88691354  0.56355906  0.07584395  0.98462325 -0.03031861
    1.0252587   0.03306922  1.034445  ]
  [ 0.89866245 -0.43343142  0.16428621  0.93683076  0.05814175
    0.98890686 -0.0083546   1.0293987 ]
  [ 0.13673374 -1.0338182   0.31809708  0.94399065  0.07854578
    0.9971316   0.01494338  1.0257467 ]]]


# Add Positional Encoding

In [ ]:
positional_aware_embeddings = word_embeddings + PE[tf.newaxis, :]

print("Position-aware Embeddings")
print(positional_aware_embeddings.numpy())

Position-aware Embeddings
[[[ 0.01029658  1.0176995  -0.03379195  0.96560216 -0.03747253
    0.98901063  0.04731724  1.0166713 ]
  [ 0.88691354  0.56355906  0.07584395  0.98462325 -0.03031861
    1.0252587   0.03306922  1.034445  ]
  [ 0.89866245 -0.43343142  0.16428621  0.93683076  0.05814175
    0.98890686 -0.0083546   1.0293987 ]
  [ 0.13673374 -1.0338182   0.31809708  0.94399065  0.07854578
    0.9971316   0.01494338  1.0257467 ]]]


# Multi Head Attention

In [ ]:
attention_layer = MultiHeadAttention(
    num_heads = 2,
    key_dim=embedding_dim
)

# Apply Self Attention

In [ ]:
attention_output = attention_layer(
    query = positional_aware_embeddings,
    value = positional_aware_embeddings,
    key = positional_aware_embeddings
)

print(attention_output.shape)
print("Contextualized Embeddings:")
print(attention_output.numpy())

(1, 4, 8)
Contextualized Embeddings:
[[[-0.41608268  0.32740512  0.11620261 -0.4101494   0.08293366
    0.09745293 -0.35729754  0.24149401]
  [-0.4174282   0.33188933  0.11351471 -0.41423205  0.08633526
    0.09129258 -0.3517406   0.24241301]
  [-0.41745514  0.3315113   0.10968488 -0.41839522  0.08586149
    0.09157248 -0.3504081   0.24773668]
  [-0.41627     0.32713574  0.10753576 -0.41951242  0.08228071
    0.09739377 -0.35389268  0.2530055 ]]]
